In [ ]:
!pip install pynput

In [ ]:
import numpy as np
import pandas as pd
import math
import time
from collections import deque
from pynput import keyboard
from pynput.keyboard import Key
import pygame
pygame.init() 

In [ ]:

ppmm = 142 / 25.4 # 142 ppi of current display

height = int(ppmm*50) # 50mm box requirement
width = int(ppmm*180)

surface = pygame.display.set_mode((width, height))  # 50mm x 180mm
pygame.display.set_caption("A* Search Visualization")

obstacle_color = (255, 192, 203)
char_color = (255, 80, 200)

# visualization colors
visited_color = (0, 0, 255)
path_color = (255, 255, 0)
start_color = (0, 255, 0)
goal_color = (255, 0, 0)

# fill free space with black background
surface.fill((0, 0, 0))

# border
# these are the mathematical models of the letter geometry for the obstacle space
# J
J_rect = pygame.draw.rect(surface, obstacle_color, pygame.Rect(117, 39, 54, 122), width=30) # basic geometry of rectangle that forms "J"
J_arc = pygame.draw.arc(surface, obstacle_color, pygame.Rect(49, 89, 122, 122), start_angle=2.90, stop_angle=0, width=60) 
# S
S_arc = pygame.draw.arc(surface, obstacle_color, pygame.Rect(169, 39, 112, 112), start_angle=0.1, stop_angle=4.81239, width=60)
S_arc_2 = pygame.draw.arc(surface, obstacle_color, pygame.Rect(169, 94, 122, 122), start_angle=-3.24, stop_angle=1.97, width=60)
# 4 432
four_rect = pygame.draw.rect(surface, obstacle_color, pygame.Rect(274, 39, 54, 97), width=30)
four_rect2 = pygame.draw.rect(surface, obstacle_color, pygame.Rect(274, 89, 97, 54), width=30)
four_rect3 = pygame.draw.rect(surface, obstacle_color, pygame.Rect(324, 39, 54, 172), width=30)
# 4
four_rect4 = pygame.draw.rect(surface, obstacle_color, pygame.Rect(374, 39, 54, 97), width=30)
four_rect5 = pygame.draw.rect(surface, obstacle_color, pygame.Rect(374, 89, 97, 54), width=30)
four_rect6 = pygame.draw.rect(surface, obstacle_color, pygame.Rect(424, 39, 54, 172), width=30)
# 3
three_rect = pygame.draw.arc(surface, obstacle_color, pygame.Rect(464, 39, 112, 112), start_angle=4.71239, stop_angle=2.85619, width=60)
three_rect2 = pygame.draw.rect(surface, obstacle_color, pygame.Rect(494, 97, 62, 54), width=30)
three_rect3 = pygame.draw.arc(surface, obstacle_color, pygame.Rect(464, 99, 112, 112), start_angle=3.42699, stop_angle=1.5708, width=60)
# 2
two_rect = pygame.draw.rect(surface, obstacle_color, pygame.Rect(589, 39, 102, 54), width=30)
two_rect2 = pygame.draw.rect(surface, obstacle_color, pygame.Rect(649, 39, 54, 114), width=30)
two_rect3 = pygame.draw.rect(surface, obstacle_color, pygame.Rect(624, 99, 54, 54), width=30)
two_rect4 = pygame.draw.rect(surface, obstacle_color, pygame.Rect(599, 129, 54, 54), width=30)
two_rect5 = pygame.draw.rect(surface, obstacle_color, pygame.Rect(579, 159, 142, 54), width=30)

# characters
# J
J_rect = pygame.draw.rect(surface, char_color, pygame.Rect(128, 50, 32, 100), width=30)
J_arc = pygame.draw.arc(surface, char_color, pygame.Rect(60, 100, 100, 100), start_angle=3.1415, stop_angle=0, width=30)
# S
S_arc = pygame.draw.arc(surface, char_color, pygame.Rect(180, 50, 90, 90), start_angle=0.5, stop_angle=4.71239, width=30)
S_arc_2 = pygame.draw.arc(surface, char_color, pygame.Rect(180, 110, 90, 90), start_angle=-2.64, stop_angle=1.57, width=30)
# 4 432
four_rect = pygame.draw.rect(surface, char_color, pygame.Rect(285, 50, 32, 75), width=30)
four_rect2 = pygame.draw.rect(surface, char_color, pygame.Rect(285, 100, 75, 32), width=30)
four_rect3 = pygame.draw.rect(surface, char_color, pygame.Rect(335, 50, 32, 150), width=30)
# 4
four_rect4 = pygame.draw.rect(surface, char_color, pygame.Rect(385, 50, 32, 75), width=30)
four_rect5 = pygame.draw.rect(surface, char_color, pygame.Rect(385, 100, 75, 32), width=30)
four_rect6 = pygame.draw.rect(surface, char_color, pygame.Rect(435, 50, 32, 150), width=30)
# 3
three_rect = pygame.draw.arc(surface, char_color, pygame.Rect(475, 50, 90, 90), start_angle=4.71239, stop_angle=2.35619, width=30)
three_rect2 = pygame.draw.rect(surface, char_color, pygame.Rect(505, 108, 40, 32), width=30)
three_rect3 = pygame.draw.arc(surface, char_color, pygame.Rect(475, 110, 90, 90), start_angle=3.92699, stop_angle=1.5708, width=30)
# 2
two_rect = pygame.draw.rect(surface, char_color, pygame.Rect(600, 50, 80, 32), width=30)
two_rect2 = pygame.draw.rect(surface, char_color, pygame.Rect(660, 50, 32, 92), width=30)
two_rect3 = pygame.draw.rect(surface, char_color, pygame.Rect(635, 110, 32, 32), width=30)
two_rect4 = pygame.draw.rect(surface, char_color, pygame.Rect(610, 140, 32, 32), width=30)
two_rect5 = pygame.draw.rect(surface, char_color, pygame.Rect(590, 170, 120, 32), width=30)

# show the map once after drawing obstaclesd
pygame.display.flip()
 

# ─────────────────────────────────────────────────────────────────────────────
# KEY HELPER: pump events so the window stays responsive while waiting for user input in the terminal
def pump():
    """Call this regularly to keep the pygame window alive."""
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            pygame.quit()
            raise SystemExit


# ─────────────────────────────────────────────────────────────────────────────
# OBSTACLE CHECK 
def is_free_space(x, y):
    if not (0 < x < width and 0 < y < height): # check if point is within bounds of the surface
        return False
    color = surface.get_at((int(x), int(y)))[:3]
    return color in [(0, 0, 0), visited_color, start_color, goal_color, path_color]# check if point isn't an obstacle


# ─────────────────────────────────────────────────────────────────────────────
# USER INPUT
def get_valid_coord(label):
    while True:
        pump()   # keep window alive while waiting for input
        try:
            x     = int(input(f"[{label}] Enter x: "))
            y     = int(input(f"[{label}] Enter y: "))
            py = height - y # convert Cartesian y → pygame y (inverted)
            theta = int(input(f"[{label}] Enter theta (multiple of 30°): "))
        except ValueError:
            print(" Enter integers only.")
            continue
        if not (0 < x < width):
            print(f" x must be 1–{width-1}")
            continue
        if not (0 < py < height):
            print(f" y must be 1–{height-1}")
            continue
        if theta % 30 != 0:
            print(" Theta must be a multiple of 30.")
            continue
        print("Checking point:", x, py, "theta:", theta)
        if not is_free_space(x, py):
            print(" Point is inside an obstacle.")
            continue
        
        # convert user (Cartesian) → pygame
        return (x, py, theta % 360)


# ─────────────────────────────────────────────────────────────────────────────
# Get user inputs - start, goal, robot radius, clearance, step size
print("Enter START state:")
start_state = get_valid_coord("START")

print("\nEnter GOAL state:")
goal_state  = get_valid_coord("GOAL")
robot_radius = int(input("Enter robot radius: "))
clearance = int(input("Enter clearance: "))

step_size = 0
while not (1 <= step_size <= 10):
    pump()
    try:   
        step_size = int(input("\nStep size L (1–10): "))
    except ValueError: 
        pass
    if not (1 <= step_size <= 10):
        print("  ✗ Must be 1–10.")

print(f"\n✓ Start={start_state}  Goal={goal_state}  L={step_size}\n")
L = step_size

# ─────────────────────────────────────────────────────────────────────────────
# Movement functions
available_turns = [-60, -30, 0, 30, 60]

def apply_action(state, turn_deg, L):
    x, y, theta = state
    new_theta = (theta + turn_deg) % 360 # keep orientation 0-359
    rad = math.radians(new_theta) # convert to radians for trig functions
    new_x = x + L * math.cos(rad)
    new_y = y - L * math.sin(rad)#   y-axis flipped in pygame

    # Collision checking each small steps along the path
    steps = max(1, int(L * 2))
    for t in range(steps + 1): # check points along the path at intervals of 0.5 units
        frac = t / steps # fraction of the way along the path
        ix = x + frac * (new_x - x) 
        iy = y + frac * (new_y - y)
        if not is_free_space(int(round(ix)), int(round(iy))): # mid-path obstacle
            return None # invalid move if path hits obstacle

    snapped_theta = int(round(new_theta / 30)) * 30 % 360 # snap to nearest multiple of 30 degrees
    return (int(round(new_x)), int(round(new_y)), snapped_theta)

def action_minus_60(state, L):
    return apply_action(state, -60, L)

def action_minus_30(state, L):
    return apply_action(state, -30, L)

def action_0(state, L):
    return apply_action(state, 0, L)

def action_plus_30(state, L):
    return apply_action(state, 30, L)

def action_plus_60(state, L):
    return apply_action(state, 60, L)

actions = [action_minus_60, action_minus_30, action_0, action_plus_30, action_plus_60 ]

def get_neighbors(state, L):
    results = []
    for action in actions:
        ns = action(state, L)
        if ns is not None:
            results.append((ns, float(L)))
    return results

def is_goal_reached(state, goal_state, pos_threshold=1.5, theta_threshold=30):
    dx = state[0] - goal_state[0]
    dy = state[1] - goal_state[1]
    dist = math.sqrt(dx*dx + dy*dy)

    dtheta = abs(state[2] - goal_state[2]) % 360
    dtheta = min(dtheta, 360 - dtheta)

    return dist <= pos_threshold and dtheta <= theta_threshold
def angle_diff(a, b):
    diff = abs(a - b) % 360
    return min(diff, 360 - diff)

def is_duplicate_node(n1, n2):   # checks if duplicated node : 1. distance < 0.5 units, 2. angle difference < 30 degrees
    thetha_threshold = 30
    euclidean_threshold = 0.5
    
     # distance check btw positions
    dx = n1[0] - n2[0]
    dy = n1[1] - n2[1]
    euclidean_threshold = math.sqrt(dx*dx + dy*dy)
    # angle check btw orientations
    diff = abs(n1[2]- n2[2]) % 360
    thetha_threshold = min(diff, 360 - diff) # angle wrap around check ex:0,350 = 10 degree difference, not 350
    return euclidean_threshold < 0.5 and thetha_threshold < 30
def state_to_key(state):
    x, y, theta = state
    x_key = round(x * 2) / 2.0
    y_key = round(y * 2) / 2.0
    theta_key = (round(theta / 30) * 30) % 360
    return (x_key, y_key, theta_key)
#----------------------------------------------------------------


# conversion for movement attempts
def tuple_to_list(state):
    return list(state) 

def list_to_tuple(state):
    return tuple(state) 


# surface = pygame.display.set_mode((900, 250))  # window
robot_radius = 5
# 
# Moving the robot
# these functions provide the 8 degrees of freedom for the point robot
def move_left(state, row, col):

    original_state = state # original position is saved

    if col - 1 > 0:

        state = tuple_to_list(state) # conversion for easier handling

        state[1] = state[1]-1 
        
        state = list_to_tuple(state) # new position is updated
    # pygame.display.flip()

    new_color = surface.get_at((state[1], state[0])) # new pixel value is recorded

    if new_color[:3] == obstacle_color: # if the new pixel position is the same color as the defined characters then the pixel does not update
        state = original_state
    
    return state # returns either updated position or same position
    
def move_right(state, row, col):
    original_state = state

    if col + 1 < width:
        state = tuple_to_list(state) 
        state[1] = state[1]+1 
        state = list_to_tuple(state) 
    # pygame.display.flip()
    
    new_color = surface.get_at((state[1], state[0]))

    if new_color[:3] == obstacle_color:
        state = original_state
    return state

def move_up(state, row, col):
    original_state = state

    if row - 1 > 0:
        state = tuple_to_list(state)
        state[0] = state[0]-1
        state = list_to_tuple(state)
   
    new_color = surface.get_at((state[1], state[0]))

    if new_color[:3] == obstacle_color:
        state = original_state
    return state

def move_down(state, row, col):
    original_state = state

    if row + 1 < height:
        state = tuple_to_list(state)
        state[0] = state[0]+1
        state = list_to_tuple(state)
    
    new_color = surface.get_at((state[1], state[0]))

    if new_color[:3] == obstacle_color:
        state = original_state
    return state

def move_diag_up_left(state, row, col):
    original_state = state

    if col - 1 > 0 and row - 1 > 0:
        state = tuple_to_list(state) 
        state[0] = state[0]-1 
        state[1] = state[1]-1 
        state = list_to_tuple(state) 
    
    new_color = surface.get_at((state[1], state[0]))

    if new_color[:3] == obstacle_color:
        state = original_state
    return state

def move_diag_up_right(state, row, col):
    original_state = state
    
    if col + 1 < width and row - 1 > 0:
        state = tuple_to_list(state)
        state[0] = state[0]-1
        state[1] = state[1]+1
        state = list_to_tuple(state)
    
    new_color = surface.get_at((state[1], state[0]))

    if new_color[:3] == obstacle_color:
        state = original_state
    return state

def move_diag_down_left(state, row, col):
    original_state = state

    if row + 1 < height and col - 1 > 0:
        state = tuple_to_list(state)
        state[0] = state[0]+1
        state[1] = state[1]-1
        state = list_to_tuple(state)
    
    new_color = surface.get_at((state[1], state[0]))

    if new_color[:3] == obstacle_color:
        state = original_state
    return state

def move_diag_down_right(state, row, col):
    original_state = state

    if row + 1 < height and col + 1 < width:
        state = tuple_to_list(state)
        state[0] = state[0]+1
        state[1] = state[1]+1
        state = list_to_tuple(state)
    
    new_color = surface.get_at((state[1], state[0]))

    if new_color[:3] == obstacle_color:
        state = original_state
    return state


closedList = []
parents = {}
children = {}
child_to_parent = {}
child_to_parent_index = {}

start_state_x = start_state[0]
start_state_y = start_state[1]

goal_state_x = goal_state[0]
goal_state_y = goal_state[1]

node_to_f_score = {start_state: np.sqrt((start_state_x - goal_state_x)**2 + (start_state_y - goal_state_y)**2)}
node_to_g_score = {start_state: 0}

f_score_node_x = deque()
f_score_a = deque()
f_score_b = deque()
f_score_c = deque()
f_score_d = deque()
f_score_e = deque()
f_score_f = deque()
f_score_g = deque()
f_score_h = deque()


# in this function the updated states are compared to the closed list for confirmed movement
# if the attempted move is the same as the current position of in the closed list then the position is not updated
def getPossibleMoves(new_node_x, row, col, counter, L):
    neighbors = get_neighbors(new_node_x, L)
    
    if not neighbors:
        return
    # store current node
    parents[counter] = new_node_x    

    # current node g score
    if new_node_x not in node_to_g_score:
        node_to_g_score[new_node_x] = 0
    g_x = node_to_g_score[new_node_x]

    # loop through however many valid neighbors exist
    for neighbor, move_cost in neighbors:
        if neighbor in closedList:
            continue

        neighbor_key = state_to_key(neighbor)
        if neighbor_key in visited_states:
            continue

        # tentative path cost through current node
        tentative_g = g_x + move_cost

        # if neighbor is new or found with lower cost, update it
        if neighbor not in node_to_g_score or tentative_g < node_to_g_score[neighbor]:
            node_to_g_score[neighbor] = tentative_g

            h = np.sqrt((goal_state_x - neighbor[0])**2 + (goal_state_y - neighbor[1])**2)
            node_to_f_score[neighbor] = tentative_g + h

            child_to_parent[neighbor] = new_node_x
            child_to_parent_index[neighbor] = counter
            
            visited_states.add(neighbor_key)

            if neighbor not in openList:
                openList.append(neighbor)
    return


openList = deque([start_state]) # open list begins with start state to initialize the loop
closedList = set()
visited_states = set()
visited_states.add(state_to_key(start_state))

counter = 0
end = 0

start_time = time.time()

# this is the main loop
# the first open list state is popped and compared to the goal state
# the check goal value is passed on to evaluate next moves
# if the check goal is equal to the goal state then the loop is broken
while node_to_f_score:
    pump()

    print("node to f score before check goal: ", node_to_f_score)
        
    check_goal_item = min(node_to_f_score.values())

    keys = [key for key, val in node_to_f_score.items() if val == check_goal_item]
    check_goal = keys[0]
    print("keys: ", keys)

    del node_to_f_score[check_goal]
    
    # if node explored, skip it
    if check_goal in closedList:
        continue 
    # remove from open list too
    if check_goal in openList:
        openList.remove(check_goal)

    if is_goal_reached(check_goal, goal_state):
        goal_state = check_goal   # snap goal to reached node
        parents[counter] = check_goal
        end = 1
        print("Goal reached:", check_goal)
        break

    closedList.add(check_goal) # add looked at node to closed list
     
    a, b = check_goal[0], check_goal[1]
    print("check_goal before get possible moves: ", check_goal)
    getPossibleMoves(check_goal, a, b, counter, L)
    
    # visualize visited node on pygame
    x, y, _ = check_goal
    pygame.draw.circle(surface, visited_color, (x, y), 1)

    # update window every few nodes so it does not slow too much
    if counter % 300 == 0:
        pygame.display.update()
        pump()

    

    if counter % 20000 == 0:
        print("Iterations: ", counter)
        print("Length of openList: ", len(openList))
        print("CLosed List: \n", len(closedList))

    counter += 1

if end == 1:
    print("SUCCESS \n SUCCESS \n SUCCESS \n SUCCESS")
else:
    print("Search ended without reaching the goal.")


print("end: ", end)
print("child to parent: ", child_to_parent)

# the shortest path is discovered using the child to parent dict from before
# the dict is reversed to account for the start of the search beginning at the end of the dict
if end == 1:
    shortest_path = [goal_state]
    current = goal_state

    while current != start_state:
        current = child_to_parent[current]
        shortest_path.append(current)

    shortest_path.reverse()

    # redraw start and goal first 
    pygame.draw.circle(surface, start_color, (start_state[0], start_state[1]), 5)
    pygame.draw.circle(surface, goal_color, (goal_state[0], goal_state[1]), 5)
 
    # draw final shortest path
    for i in range(len(shortest_path) - 1):
        pump()
        x1, y1, _ = shortest_path[i]
        x2, y2, _ = shortest_path[i + 1]
        pygame.draw.line(surface, path_color, (x1, y1), (x2, y2), 2)
        pygame.display.update()

    # redraw start and goal so they stay visible
    pygame.draw.circle(surface, start_color, (start_state[0], start_state[1]), 5)
    pygame.draw.circle(surface, goal_color, (goal_state[0], goal_state[1]), 5)
    pygame.display.update()

    print("Shortest path:", shortest_path)

print("--- %s seconds ---" % (time.time() - start_time)) # time for full search and shortest path is recorded for analysis

# keep pygame window open after search so result can be seen
print("Close the pygame window to exit.")

while True:
    pump()
    pygame.display.update()
    pygame.time.delay(30)